# 03 - World Cup predictions and backtest

Train on pre-tournament history, predict every fixture chronologically, compare completed predictions with actual results, and export dated CSV/JSON artifacts.

In [ ]:
import os
import pandas as pd
from fifa_predict.pipeline import predict_tournament

offline = os.getenv("FIFA_PREDICT_OFFLINE", "").lower() in {"1", "true", "yes"}
predictions, paths, bundle, audit = predict_tournament(refresh=not offline, offline=offline)
print(f"Model: {bundle.model_version}")
print(f"Wrote: {paths[0]}")
print(f"Wrote: {paths[1]}")

In [ ]:
prediction_frame = pd.DataFrame([record.to_dict() for record in predictions])
upcoming = None
if not prediction_frame.empty:
    probability_sum = prediction_frame[["home_win_probability", "draw_probability", "away_win_probability"]].sum(axis=1)
    assert probability_sum.between(0.999999, 1.000001).all()
    completed = prediction_frame[prediction_frame.actual_result.notna()]
    print(f"Completed-match accuracy: {completed.prediction_correct.mean():.1%}" if len(completed) else "No completed matches yet")
    print(f"Player context applied: {prediction_frame.player_context_applied.sum()} of {len(prediction_frame)} fixtures")
    display(completed[["home_team", "away_team", "home_win_probability", "draw_probability", "away_win_probability", "predicted_result", "actual_home_score", "actual_away_score", "actual_result", "prediction_correct", "player_context_applied", "home_expected_xi_quality", "away_expected_xi_quality", "home_lineup_confirmed", "away_lineup_confirmed"]])
else:
    print("No tournament fixtures were found at this cutoff.")

In [ ]:
if not prediction_frame.empty:
    upcoming_matches = prediction_frame[prediction_frame.actual_result.isna()].sort_values("match_date")
    upcoming = upcoming_matches.iloc[0] if len(upcoming_matches) else None
if upcoming is not None:
    print(
        f"Next: {upcoming.home_team} vs {upcoming.away_team} - "
        f"H {upcoming.home_win_probability:.1%}, "
        f"D {upcoming.draw_probability:.1%}, "
        f"A {upcoming.away_win_probability:.1%}"
    )